# Detecting and Preventing Data Poisoning Attacks
**Course:** AI354 — Responsible AI  |  **Project:** 10  
**Team:** I23MA002 Devesh Singh Chauhan  ·  U23EE102 Priyanshu Sanjay Gawas  
**Department:** Artificial Intelligence, SVNIT

---

## Overview
This notebook demonstrates **data poisoning attacks** on ML models and defenses against them.

**What we do:**
- Inject poisoned (label-flipped) samples into training data
- Detect them using Z-score filtering and Isolation Forest
- Measure accuracy loss from the attack and recovery from the defense

**Detection methods used:**
- **Z-Score Analysis** — flags samples where any feature deviates > 2.5σ from the mean
- **Isolation Forest** — unsupervised anomaly detection via random partitioning

**Datasets:**
- **Image model:** `sklearn.datasets.load_digits` — 1,797 samples of 8×8 handwritten digit images (10 classes), no download required
- **Text model:** Synthetic corpus of 600 documents (300 “Tech” + 300 “Sports”), features extracted with TF-IDF (500 dimensions)

> All plots are saved to the `images/` folder.

## Section 0 — Imports & Setup

In [1]:
import os, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')          # save to disk only, no display
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from sklearn.datasets               import load_digits
from sklearn.model_selection        import train_test_split
from sklearn.preprocessing          import StandardScaler
from sklearn.linear_model           import LogisticRegression
from sklearn.ensemble               import IsolationForest
from sklearn.metrics                import accuracy_score, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.stats                    import zscore
from datetime                       import datetime

os.makedirs('images', exist_ok=True)
np.random.seed(42)
print('Imports OK. Plots will be saved to ./images/')

Imports OK. Plots will be saved to ./images/


## Section 1 — Z-Score Statistical Detection
Flag samples where any feature has |z| > 2.5. Fast first-pass detection.

In [2]:
# Synthetic dataset: 300 clean samples + 15 injected outliers
np.random.seed(42)
n_samples, n_poison = 300, 15

feature_1 = np.random.normal(0.0, 1.0, n_samples)
feature_2 = np.random.normal(5.0, 1.5, n_samples)
labels    = np.random.randint(0, 2, n_samples)

# Poison: extreme values far outside normal range
poison_f1     = np.random.uniform(8, 12, n_poison)
poison_f2     = np.random.uniform(-5, -3, n_poison)
poison_labels = np.random.randint(0, 2, n_poison)

X_raw      = np.column_stack([np.concatenate([feature_1, poison_f1]),
                               np.concatenate([feature_2, poison_f2])])
all_labels = np.concatenate([labels, poison_labels])

print(f'Dataset: {n_samples} clean + {n_poison} poisoned = {len(X_raw)} total')

Dataset: 300 clean + 15 poisoned = 315 total


In [3]:
# Z-score filter
threshold       = 2.5
z_scores        = np.abs(zscore(X_raw, axis=0))
suspicious_mask = np.any(z_scores > threshold, axis=1)

true_positives    = suspicious_mask[n_samples:].sum()
false_positives_z = suspicious_mask[:n_samples].sum()

print(f'Threshold              : {threshold}')
print(f'Poison caught          : {true_positives}/{n_poison}  ({true_positives/n_poison*100:.0f}%)')
print(f'False positives (clean): {false_positives_z}')

Threshold              : 2.5
Poison caught          : 15/15  (100%)
False positives (clean): 0


In [4]:
# Save Z-score detection plot -> images/zscore_detection.png
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(X_raw[:n_samples,0], X_raw[:n_samples,1], c='steelblue', alpha=0.5, label='Clean', s=40)
ax.scatter(X_raw[n_samples:,0], X_raw[n_samples:,1], c='red', marker='X', s=120, label='Poisoned', zorder=5)
ax.set_title('Dataset: Clean vs Poisoned', fontsize=13, fontweight='bold')
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
idx = np.arange(len(X_raw))
clean_unflagged = ~suspicious_mask & (idx < n_samples)
ax.scatter(X_raw[clean_unflagged,0], X_raw[clean_unflagged,1],
           c='steelblue', alpha=0.5, label='Clean (not flagged)', s=40)
fp_mask = suspicious_mask & (idx < n_samples)
ax.scatter(X_raw[fp_mask,0], X_raw[fp_mask,1], c='orange', marker='s', s=80, label='False positive')
tp_mask = suspicious_mask[n_samples:]
ax.scatter(X_raw[n_samples:][tp_mask,0],  X_raw[n_samples:][tp_mask,1],
           c='red', marker='X', s=120, label='Poison caught', zorder=5)
ax.scatter(X_raw[n_samples:][~tp_mask,0], X_raw[n_samples:][~tp_mask,1],
           c='purple', marker='X', s=120, label='Missed poison', zorder=5)
ax.set_title('Z-Score Detection Results', fontsize=13, fontweight='bold')
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Section 1: Z-Score Statistical Detection', fontsize=15, y=1.02)
plt.savefig('images/zscore_detection.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved: images/zscore_detection.png')

Saved: images/zscore_detection.png


## Section 2 — Isolation Forest Detection
Unsupervised anomaly detection via random partitioning. Catches cluster-level anomalies that Z-score misses.

In [5]:
# Isolation Forest on same synthetic dataset
iso_forest       = IsolationForest(contamination=0.05, random_state=42, n_estimators=100)
iso_predictions  = iso_forest.fit_predict(X_raw)
iso_anomaly_mask = iso_predictions == -1

iso_tp            = iso_anomaly_mask[n_samples:].sum()
false_positives_iso = iso_anomaly_mask[:n_samples].sum()
scores            = iso_forest.decision_function(X_raw)

print(f'Total flagged          : {iso_anomaly_mask.sum()}')
print(f'Poison caught          : {iso_tp}/{n_poison}  ({iso_tp/n_poison*100:.0f}%)')
print(f'False positives (clean): {false_positives_iso}')
print(f'Avg score clean data   : {scores[:n_samples].mean():.3f}')
print(f'Avg score poison data  : {scores[n_samples:].mean():.3f}  (more negative = more anomalous)')

Total flagged          : 16
Poison caught          : 12/15  (80%)
False positives (clean): 4
Avg score clean data   : 0.209
Avg score poison data  : -0.040  (more negative = more anomalous)


In [6]:
# Save comparison plot -> images/detection_comparison.png
methods   = ['Z-Score', 'Isolation Forest']
det_rates = [true_positives/n_poison*100, iso_tp/n_poison*100]
fp_counts = [false_positives_z, false_positives_iso]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(methods, det_rates, color=['#e74c3c','#2ecc71'], edgecolor='black', width=0.5)
axes[0].set_ylabel('Detection Rate (%)'); axes[0].set_ylim(0, 110)
axes[0].set_title('Poison Detection Rate', fontsize=13, fontweight='bold')
for i, v in enumerate(det_rates):
    axes[0].text(i, v+2, f'{v:.1f}%', ha='center', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(methods, fp_counts, color=['#e67e22','#3498db'], edgecolor='black', width=0.5)
axes[1].set_ylabel('Number of False Alarms')
axes[1].set_title('False Positives (Clean data wrongly flagged)', fontsize=13, fontweight='bold')
for i, v in enumerate(fp_counts):
    axes[1].text(i, v+0.3, str(v), ha='center', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.suptitle('Section 2: Detection Method Comparison', fontsize=15, y=1.02)
plt.savefig('images/detection_comparison.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved: images/detection_comparison.png')

print(f'\nZ-Score      : {true_positives}/{n_poison} caught, {false_positives_z} FP')
print(f'Iso Forest   : {iso_tp}/{n_poison} caught, {false_positives_iso} FP')
print(f'Combined (OR): {max(true_positives,iso_tp)}/{n_poison} caught')

Saved: images/detection_comparison.png

Z-Score      : 15/15 caught, 0 FP
Iso Forest   : 12/15 caught, 4 FP
Combined (OR): 15/15 caught


## Section 3 — Image Model: Attack & Defense
> **Dataset:** `sklearn` Digits — 1,797 samples, 8×8 pixel images, 10 classes (0–9).  
> **Attack:** 10% random label-flip.  **Defense:** Isolation Forest pre-filtering.

In [7]:
# Load and split digits dataset
digits = load_digits()
X_img, y_img = digits.data, digits.target

X_img_train, X_img_test, y_img_train, y_img_test = train_test_split(
    X_img, y_img, test_size=0.2, random_state=42
)

scaler        = StandardScaler()
X_img_train_s = scaler.fit_transform(X_img_train)
X_img_test_s  = scaler.transform(X_img_test)

print(f'Train / Test : {X_img_train.shape[0]} / {X_img_test.shape[0]} samples')
print(f'Image shape  : {digits.images[0].shape}  (8x8 = 64 features per image)')
print(f'Classes      : {np.unique(y_img).tolist()}')

Train / Test : 1437 / 360 samples
Image shape  : (8, 8)  (8x8 = 64 features per image)
Classes      : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [8]:
# Save sample image grid -> images/poisoned_images.png
# Shows original vs label-flipped (pixel data identical, label changes)
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for digit in range(10):
    idx = np.where(y_img_train == digit)[0][0]
    axes[0, digit].imshow(X_img_train[idx].reshape(8, 8), cmap='gray_r')
    axes[0, digit].set_title(f'Label: {digit}', fontsize=8)
    axes[0, digit].axis('off')
    axes[1, digit].imshow(X_img_train[idx].reshape(8, 8), cmap='Reds')
    axes[1, digit].set_title(f'Faked->{(digit+1)%10}', fontsize=8, color='red')
    axes[1, digit].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=10)
axes[1, 0].set_ylabel('Poisoned', fontsize=10)
plt.suptitle('Digit Images: Original Labels vs Poisoned (Flipped) Labels', fontsize=12)
plt.tight_layout()
plt.savefig('images/poisoned_images.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved: images/poisoned_images.png')
print('Note: pixel data is unchanged; only the label is secretly flipped.')

Saved: images/poisoned_images.png
Note: pixel data is unchanged; only the label is secretly flipped.


In [9]:
# Inject 10% label-flip attack
poison_rate    = 0.10
n_img_poison   = int(poison_rate * len(y_img_train))
img_poison_idx = np.random.choice(len(y_img_train), n_img_poison, replace=False)

y_img_poisoned = y_img_train.copy()
for idx in img_poison_idx:
    orig = y_img_poisoned[idx]
    y_img_poisoned[idx] = np.random.choice([c for c in range(10) if c != orig])

print(f'Poisoned {n_img_poison}/{len(y_img_train)} labels ({poison_rate*100:.0f}% attack rate)')

Poisoned 143/1437 labels (10% attack rate)


In [10]:
# Train baseline, poisoned, and defended models

# 1. Baseline (clean data)
clf_img_clean = LogisticRegression(max_iter=2000, random_state=42)
clf_img_clean.fit(X_img_train_s, y_img_train)
acc_img_clean = accuracy_score(y_img_test, clf_img_clean.predict(X_img_test_s))

# 2. Poisoned (attacked data)
clf_img_poison = LogisticRegression(max_iter=2000, random_state=42)
clf_img_poison.fit(X_img_train_s, y_img_poisoned)
acc_img_poison = accuracy_score(y_img_test, clf_img_poison.predict(X_img_test_s))

# 3. Defended: Isolation Forest removes anomalies, then train on clean labels
iso_img       = IsolationForest(contamination=0.10, random_state=42)
iso_img_flags = iso_img.fit_predict(X_img_train_s)
X_img_def     = X_img_train_s[iso_img_flags == 1]
y_img_def     = y_img_train[iso_img_flags == 1]    # original labels, not poisoned

clf_img_defended = LogisticRegression(max_iter=2000, random_state=42)
clf_img_defended.fit(X_img_def, y_img_def)
acc_img_defended = accuracy_score(y_img_test, clf_img_defended.predict(X_img_test_s))

print('-' * 50)
print(f'  Baseline (clean)   : {acc_img_clean*100:.2f}%')
print(f'  Poisoned (10% flip): {acc_img_poison*100:.2f}%  (drop: -{(acc_img_clean-acc_img_poison)*100:.2f}%)')
print(f'  Defended (filtered): {acc_img_defended*100:.2f}%  (recovery: +{(acc_img_defended-acc_img_poison)*100:.2f}%)')
print('-' * 50)

--------------------------------------------------
  Baseline (clean)   : 97.22%
  Poisoned (10% flip): 90.83%  (drop: -6.39%)
  Defended (filtered): 96.39%  (recovery: +5.56%)
--------------------------------------------------


In [11]:
# Save accuracy bar chart + confusion matrix -> images/image_model_results.png
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

model_names = ['Baseline\n(Clean)', 'Poisoned\n(Attacked)', 'Defended\n(Filtered)']
img_accs    = [acc_img_clean*100, acc_img_poison*100, acc_img_defended*100]
colors      = ['#27ae60', '#e74c3c', '#2980b9']

bars = axes[0].bar(model_names, img_accs, color=colors, edgecolor='black', width=0.5)
axes[0].set_ylim(0, 105); axes[0].set_ylabel('Test Accuracy (%)', fontsize=11)
axes[0].set_title('Image Model: Attack vs Defense\n(Digits Dataset)', fontsize=12, fontweight='bold')
for bar, acc in zip(bars, img_accs):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'{acc:.1f}%', ha='center', fontsize=11, fontweight='bold')
axes[0].axhline(y=acc_img_clean*100, color='green', linestyle='--', alpha=0.5, label='Baseline')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

cm = confusion_matrix(y_img_test, clf_img_poison.predict(X_img_test_s))
im = axes[1].imshow(cm, interpolation='nearest', cmap='Blues')
axes[1].set_title('Poisoned Model: Confusion Matrix\n(darker = more predictions)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Label'); axes[1].set_ylabel('True Label')
axes[1].set_xticks(range(10)); axes[1].set_yticks(range(10))
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.savefig('images/image_model_results.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved: images/image_model_results.png')

Saved: images/image_model_results.png


## Section 4 — Text Model: Attack & Defense
> **Dataset:** Synthetic corpus — 600 documents, 300 per class (“Tech” vs “Sports”).  
> Features: TF-IDF with 500 dimensions.  **Attack:** 20% label-flip.  **Defense:** Isolation Forest.

In [12]:
# Build synthetic text corpus (no download needed)
tech_words   = ['algorithm','neural','network','data','model','training','python','code',
                'software','computer','machine','learning','deep','layer','feature',
                'dataset','accuracy','loss','gradient','optimization','tensor','batch','epoch']
sports_words = ['goal','team','player','match','score','league','tournament','championship',
                'win','season','game','athlete','stadium','coach','referee','penalty',
                'defense','offense','trophy','final','quarter','overtime']
noise_words  = ['the','a','is','was','it','of','in','and','to','that']

def make_doc(cat_words, length=40):
    words = ([np.random.choice(cat_words) for _ in range(length//2)] +
             [np.random.choice(noise_words) for _ in range(length//2)])
    np.random.shuffle(words)
    return ' '.join(words)

N = 300
all_docs   = [make_doc(tech_words) for _ in range(N)] + [make_doc(sports_words) for _ in range(N)]
all_labels = np.array([0]*N + [1]*N)  # 0=Tech, 1=Sports

shuffle_idx = np.random.permutation(len(all_docs))
all_docs    = [all_docs[i] for i in shuffle_idx]
all_labels  = all_labels[shuffle_idx]

split_at = int(0.8*len(all_docs))
train_docs, test_docs   = all_docs[:split_at], all_docs[split_at:]
y_txt_train, y_txt_test = all_labels[:split_at], all_labels[split_at:]

vectorizer  = TfidfVectorizer(max_features=500, stop_words='english')
X_txt_train = vectorizer.fit_transform(train_docs).toarray()
X_txt_test  = vectorizer.transform(test_docs).toarray()

print(f'Train / Test docs  : {len(train_docs)} / {len(test_docs)}')
print(f'TF-IDF dimensions  : {X_txt_train.shape[1]}')
print(f'Classes            : 0 = Tech, 1 = Sports')

Train / Test docs  : 480 / 120
TF-IDF dimensions  : 45
Classes            : 0 = Tech, 1 = Sports


In [13]:
# Inject 20% label-flip attack
txt_poison_rate = 0.20
n_txt_poison    = int(txt_poison_rate * len(y_txt_train))
txt_poison_idx  = np.random.choice(len(y_txt_train), n_txt_poison, replace=False)

y_txt_poisoned  = y_txt_train.copy()
y_txt_poisoned[txt_poison_idx] = 1 - y_txt_poisoned[txt_poison_idx]  # flip 0 <-> 1

print(f'Poisoned {n_txt_poison}/{len(y_txt_train)} labels ({txt_poison_rate*100:.0f}% attack rate)')

Poisoned 96/480 labels (20% attack rate)


In [14]:
# Train baseline, poisoned, and defended text models

clf_txt_clean = LogisticRegression(max_iter=1000, random_state=42)
clf_txt_clean.fit(X_txt_train, y_txt_train)
acc_txt_clean = accuracy_score(y_txt_test, clf_txt_clean.predict(X_txt_test))

clf_txt_poison = LogisticRegression(max_iter=1000, random_state=42)
clf_txt_poison.fit(X_txt_train, y_txt_poisoned)
acc_txt_poison = accuracy_score(y_txt_test, clf_txt_poison.predict(X_txt_test))

iso_txt       = IsolationForest(contamination=0.20, random_state=42)
iso_txt_flags = iso_txt.fit_predict(X_txt_train)
X_txt_def     = X_txt_train[iso_txt_flags == 1]
y_txt_def     = y_txt_train[iso_txt_flags == 1]

clf_txt_defended = LogisticRegression(max_iter=1000, random_state=42)
clf_txt_defended.fit(X_txt_def, y_txt_def)
acc_txt_defended = accuracy_score(y_txt_test, clf_txt_defended.predict(X_txt_test))

print('-' * 50)
print(f'  Baseline (clean)   : {acc_txt_clean*100:.2f}%')
print(f'  Poisoned (20% flip): {acc_txt_poison*100:.2f}%  (drop: -{(acc_txt_clean-acc_txt_poison)*100:.2f}%)')
print(f'  Defended (filtered): {acc_txt_defended*100:.2f}%  (recovery: +{(acc_txt_defended-acc_txt_poison)*100:.2f}%)')
print('-' * 50)

--------------------------------------------------
  Baseline (clean)   : 100.00%
  Poisoned (20% flip): 100.00%  (drop: -0.00%)
  Defended (filtered): 100.00%  (recovery: +0.00%)
--------------------------------------------------


In [15]:
# Save TF-IDF feature weight plot -> images/text_model_features.png
feature_names = vectorizer.get_feature_names_out()
coef          = clf_txt_clean.coef_[0]  # positive = Sports, negative = Tech
top_sports    = np.argsort(coef)[-15:][::-1]
top_tech      = np.argsort(coef)[:15]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(range(15), coef[top_sports], color='#3498db', edgecolor='black')
axes[0].set_yticks(range(15))
axes[0].set_yticklabels([feature_names[i] for i in top_sports])
axes[0].set_title('Top Words -> "Sports" Category', fontsize=12, fontweight='bold')
axes[0].set_xlabel('TF-IDF Coefficient'); axes[0].grid(axis='x', alpha=0.3)

axes[1].barh(range(15), coef[top_tech], color='#e74c3c', edgecolor='black')
axes[1].set_yticks(range(15))
axes[1].set_yticklabels([feature_names[i] for i in top_tech])
axes[1].set_title('Top Words -> "Tech" Category', fontsize=12, fontweight='bold')
axes[1].set_xlabel('TF-IDF Coefficient'); axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('Section 4: What the Text Model Learned (Clean Baseline)', fontsize=14)
plt.tight_layout()
plt.savefig('images/text_model_features.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved: images/text_model_features.png')

Saved: images/text_model_features.png


## Section 5 — Automated Alerting: `DataPoisoningDetector`
Wraps both detection methods into a reusable class with severity-graded alerts (OK / WARNING / CRITICAL).

In [16]:
class DataPoisoningDetector:
    """Combines Z-score + Isolation Forest; emits OK/WARNING/CRITICAL alerts."""

    def __init__(self, zscore_threshold=2.5, iso_contamination=0.10):
        self.zt        = zscore_threshold
        self.iso       = IsolationForest(contamination=iso_contamination, random_state=42)
        self.alert_log = []
        self._fitted   = False

    def fit(self, X):
        self.iso.fit(X); self._fitted = True
        self._log('INFO', f'Detector fitted on {X.shape[0]} samples')
        return self

    def scan(self, X, name='dataset'):
        assert self._fitted, 'Call fit() first'
        z_flag   = np.any(np.abs(zscore(X, axis=0)) > self.zt, axis=1)
        iso_flag = self.iso.predict(X) == -1
        combined = z_flag | iso_flag
        pct   = combined.sum() / len(X) * 100
        level = 'CRITICAL' if pct > 15 else 'WARNING' if pct > 8 else 'OK'
        self._log(level,
            f'[{name}] {combined.sum()}/{len(X)} flagged ({pct:.1f}%) '
            f'-- Z-score: {z_flag.sum()}, IsoForest: {iso_flag.sum()}')
        return combined

    def clean(self, X, y, name='dataset'):
        mask = self.scan(X, name)
        self._log('INFO', f'[{name}] Removed {mask.sum()} samples, {(~mask).sum()} remain')
        return X[~mask], y[~mask]

    def _log(self, level, msg):
        ts    = datetime.now().strftime('%H:%M:%S')
        entry = f'[{ts}] {level}: {msg}'
        self.alert_log.append(entry)
        print(entry)

print('DataPoisoningDetector class defined.')

DataPoisoningDetector class defined.


In [17]:
# Scan both datasets
det_img = DataPoisoningDetector(zscore_threshold=2.5, iso_contamination=0.10)
det_img.fit(X_img_train_s)
img_flags = det_img.scan(X_img_train_s, name='Image Training Set')

det_txt = DataPoisoningDetector(zscore_threshold=2.5, iso_contamination=0.20)
det_txt.fit(X_txt_train)
txt_flags = det_txt.scan(X_txt_train, name='Text Training Set')

[07:31:10] INFO: Detector fitted on 1437 samples
[07:31:10] CRITICAL: [Image Training Set] 587/1437 flagged (40.8%) -- Z-score: 586, IsoForest: 144
[07:31:10] INFO: Detector fitted on 480 samples
[07:31:10] CRITICAL: [Text Training Set] 421/480 flagged (87.7%) -- Z-score: 404, IsoForest: 96


In [18]:
# Clean and retrain final defended models
X_img_final, y_img_final = det_img.clean(X_img_train_s, y_img_train, 'Image Training Set')
X_txt_final, y_txt_final = det_txt.clean(X_txt_train,   y_txt_train, 'Text Training Set')

final_img = LogisticRegression(max_iter=2000, random_state=42)
final_img.fit(X_img_final, y_img_final)
acc_img_final = accuracy_score(y_img_test, final_img.predict(X_img_test_s))

final_txt = LogisticRegression(max_iter=1000, random_state=42)
final_txt.fit(X_txt_final, y_txt_final)
acc_txt_final = accuracy_score(y_txt_test, final_txt.predict(X_txt_test))

print(f'Final Image Model: {acc_img_final*100:.2f}%')
print(f'Final Text  Model: {acc_txt_final*100:.2f}%')

[07:31:10] CRITICAL: [Image Training Set] 587/1437 flagged (40.8%) -- Z-score: 586, IsoForest: 144
[07:31:10] INFO: [Image Training Set] Removed 587 samples, 850 remain
[07:31:10] CRITICAL: [Text Training Set] 421/480 flagged (87.7%) -- Z-score: 404, IsoForest: 96
[07:31:10] INFO: [Text Training Set] Removed 421 samples, 59 remain
Final Image Model: 93.33%
Final Text  Model: 100.00%


## Section 6 — Full Results Summary

In [19]:
# Save full summary dashboard -> images/final_summary.png
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Data Poisoning Attack & Defense - Full Results Summary',
             fontsize=16, fontweight='bold', y=1.01)

model_names = ['Baseline\n(Clean)', 'Poisoned\n(Attacked)', 'Defended\n(Filtered)']
colors      = ['#27ae60', '#e74c3c', '#2980b9']

# Image model accuracy
ax = axes[0, 0]
img_accs = [acc_img_clean*100, acc_img_poison*100, acc_img_final*100]
bars = ax.bar(model_names, img_accs, color=colors, edgecolor='black', width=0.5)
ax.set_ylim(60, 102); ax.set_ylabel('Accuracy (%)'); ax.grid(axis='y', alpha=0.3)
ax.set_title('Image Model (Digits)', fontsize=12, fontweight='bold')
for bar, acc in zip(bars, img_accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{acc:.1f}%', ha='center', fontsize=10, fontweight='bold')

# Text model accuracy
ax = axes[0, 1]
txt_accs = [acc_txt_clean*100, acc_txt_poison*100, acc_txt_final*100]
bars = ax.bar(model_names, txt_accs, color=colors, edgecolor='black', width=0.5)
ax.set_ylim(60, 105); ax.set_ylabel('Accuracy (%)'); ax.grid(axis='y', alpha=0.3)
ax.set_title('Text Model (TF-IDF)', fontsize=12, fontweight='bold')
for bar, acc in zip(bars, txt_accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{acc:.1f}%', ha='center', fontsize=10, fontweight='bold')

# Detection rates on image dataset
ax = axes[1, 0]
z_det   = np.any(np.abs(zscore(X_img_train_s, axis=0)) > 2.5, axis=1)[img_poison_idx].mean()*100
iso_det = (iso_img.predict(X_img_train_s) == -1)[img_poison_idx].mean()*100
bars = ax.bar(['Z-Score\nDetection', 'Isolation\nForest'], [z_det, iso_det],
              color=['#8e44ad','#f39c12'], edgecolor='black', width=0.4)
ax.set_ylim(0, 110); ax.set_ylabel('Detection Rate (%)'); ax.grid(axis='y', alpha=0.3)
ax.set_title('Detection Rate on Poisoned Samples\n(Image Dataset)', fontsize=12, fontweight='bold')
for bar, r in zip(bars, [z_det, iso_det]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{r:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Summary table
ax = axes[1, 1]; ax.axis('off')
table_data = [
    ['Metric',            'Image Model',                                  'Text Model'],
    ['Baseline Accuracy', f'{acc_img_clean*100:.1f}%',                   f'{acc_txt_clean*100:.1f}%'],
    ['Poisoned Accuracy', f'{acc_img_poison*100:.1f}%',                  f'{acc_txt_poison*100:.1f}%'],
    ['Defended Accuracy', f'{acc_img_final*100:.1f}%',                   f'{acc_txt_final*100:.1f}%'],
    ['Attack Drop',       f'-{(acc_img_clean-acc_img_poison)*100:.1f}%', f'-{(acc_txt_clean-acc_txt_poison)*100:.1f}%'],
    ['Defense Recovery',  f'+{(acc_img_final-acc_img_poison)*100:.1f}%', f'+{(acc_txt_final-acc_txt_poison)*100:.1f}%'],
    ['Poison Rate',       '10%',                                         '20%'],
    ['Defense Method',    'Isolation Forest',                            'Isolation Forest'],
]
tbl = ax.table(cellText=table_data[1:], colLabels=table_data[0],
               cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
for j in range(3):
    tbl[0, j].set_facecolor('#2c3e50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')
ax.set_title('Summary Table', fontsize=12, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig('images/final_summary.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved: images/final_summary.png')

Saved: images/final_summary.png


## Final Results

In [21]:
print('\n  IMAGE MODEL  (sklearn Digits, 10% label-flip attack)')
print(f'    Baseline  : {acc_img_clean*100:.2f}%')
print(f'    Attacked  : {acc_img_poison*100:.2f}%  (drop  : -{(acc_img_clean-acc_img_poison)*100:.2f}%)')
print(f'    Defended  : {acc_img_final*100:.2f}%  (recovery: +{(acc_img_final-acc_img_poison)*100:.2f}%)')

print('\n  TEXT MODEL   (Synthetic TF-IDF corpus, 20% label-flip attack)')
print(f'    Baseline  : {acc_txt_clean*100:.2f}%')
print(f'    Attacked  : {acc_txt_poison*100:.2f}%  (drop  : -{(acc_txt_clean-acc_txt_poison)*100:.2f}%)')
print(f'    Defended  : {acc_txt_final*100:.2f}%  (recovery: +{(acc_txt_final-acc_txt_poison)*100:.2f}%)')

print('\n  DETECTION   (synthetic outlier set, 15 injected poisons)')
print(f'    Z-Score          : {true_positives}/{n_poison} caught ({true_positives/n_poison*100:.0f}%), FP: {false_positives_z}')
print(f'    Isolation Forest : {iso_tp}/{n_poison} caught ({iso_tp/n_poison*100:.0f}%), FP: {false_positives_iso}')

imgs = sorted(os.listdir('images'))
print(f'\n  Saved {len(imgs)} images: {chr(10)}    ' + f'{chr(10)}    '.join(imgs))
print('=' * 55)


  IMAGE MODEL  (sklearn Digits, 10% label-flip attack)
    Baseline  : 97.22%
    Attacked  : 90.83%  (drop  : -6.39%)
    Defended  : 93.33%  (recovery: +2.50%)

  TEXT MODEL   (Synthetic TF-IDF corpus, 20% label-flip attack)
    Baseline  : 100.00%
    Attacked  : 100.00%  (drop  : -0.00%)
    Defended  : 100.00%  (recovery: +0.00%)

  DETECTION   (synthetic outlier set, 15 injected poisons)
    Z-Score          : 15/15 caught (100%), FP: 0
    Isolation Forest : 12/15 caught (80%), FP: 4

  Saved 6 images: 
    detection_comparison.png
    final_summary.png
    image_model_results.png
    poisoned_images.png
    text_model_features.png
    zscore_detection.png
